**Business Understanding of MPG Dataset**

The MPG dataset contains automobile specifications and fuel efficiency information of different vehicles.

 The main goal of this project is to predict the MPG (miles per gallon) of a vehicles using its technical features.

In [1]:
!pip install tensorflow
!pip install pymysql
!pip install sqlalchemy

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error,r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

from sqlalchemy import create_engine
import pymysql

**Loading Dataset**

In [3]:
df = sns.load_dataset('mpg')
print(df.head(3))

    mpg  cylinders  displacement  horsepower  weight  acceleration  \
0  18.0          8         307.0       130.0    3504          12.0   
1  15.0          8         350.0       165.0    3693          11.5   
2  18.0          8         318.0       150.0    3436          11.0   

   model_year origin                       name  
0          70    usa  chevrolet chevelle malibu  
1          70    usa          buick skylark 320  
2          70    usa         plymouth satellite  


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    object 
 8   name          398 non-null    object 
dtypes: float64(4), int64(3), object(2)
memory usage: 28.1+ KB
None


**Missing values**

In [5]:
df.isnull().sum()

,0
mpg,0
cylinders,0
displacement,0
horsepower,6
weight,0
acceleration,0
model_year,0
origin,0
name,0


In [6]:
df[df['horsepower'].isnull()]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
32,25.0,4,98.0,NaN,2046,19.0,71,usa,ford pinto
126,21.0,6,200.0,NaN,2875,17.0,74,usa,ford maverick
330,40.9,4,85.0,NaN,1835,17.3,80,europe,renault lecar deluxe
336,23.6,4,140.0,NaN,2905,14.3,80,usa,ford mustang cobra
354,34.5,4,100.0,NaN,2320,15.8,81,europe,renault 18i
374,23.0,4,151.0,NaN,3035,20.5,82,usa,amc concord dl


**check the number of records**

In [7]:
df.shape

(398, 9)

**drop the missing rows**

In [8]:
df = df.dropna()
df.shape

(392, 9)

In [9]:
df.columns

Index(['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
       'acceleration', 'model_year', 'origin', 'name'],
      dtype='object')

**Remove Categorical Columns**

In [10]:
df=df.drop(['origin','name','model_year'],axis=1)
df.head(3)

,mpg,cylinders,displacement,horsepower,weight,acceleration
0,18.0,8,307.0,130.0,3504,12.0
1,15.0,8,350.0,165.0,3693,11.5
2,18.0,8,318.0,150.0,3436,11.0


**Insights**

Categorical columns were removed because neural networks require nmerical input.

This preprocessing step prepares the dataset for deep learning.

**Defining Features and Target Variable**

In [11]:
X = df.drop('mpg',axis=1)
y = df['mpg']

print(X.head(3))
print(y.head(3))

   cylinders  displacement  horsepower  weight  acceleration
0          8         307.0       130.0    3504          12.0
1          8         350.0       165.0    3693          11.5
2          8         318.0       150.0    3436          11.0
0    18.0
1    15.0
2    18.0
Name: mpg, dtype: float64


**Train Test Split**

In [12]:
X_train, X_test,y_train,y_test =train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
    )

**Feature Scaling using Standard Scaler**

In [13]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

**Building Multi Layer Perceptron Feed Forward Neural Network Model**

In [14]:
model =tf.keras.models.Sequential([
    tf.keras.layers.Dense(64, activation='relu',input_shape = (X_train_scaled.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16,activation='relu'),
    tf.keras.layers.Dense(1)
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Compiling the Neural Network**

In [15]:
model.compile(optimizer='adam',
              loss='mean_squared_error',
              metrics=['mae']
)

**Model Summary**

In [16]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,009 (11.75 KB)

 Trainable params: 3,009 (11.75 KB)

 Non-trainable params: 0 (0.00 B)

**Insights**

The neural network contains multiple hidden layers.
Each layer learns complex relationships bvetween vehicle specifications and MPG values.
The model architecture helps improve prediction capability.

**Training the Neural Network**

In [17]:
history = model.fit(X_train_scaled,
                    y_train,
                    epochs =100,
                    batch_size=16,
                    verbose =1
)

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 598.4247 - mae: 23.1365
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 541.8391 - mae: 21.9351 
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 421.0513 - mae: 19.0323
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 211.1385 - mae: 12.8281
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 66.2084 - mae: 6.7479
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 50.7480 - mae: 5.5490
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 40.9694 - mae: 4.9799
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 35.3908 - mae: 4.6414
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 30.9913 - mae: 4.3034
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 27.4204 - mae: 4.0095
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 24.8633 - mae: 3.7841
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 22.9560 - mae: 3.5896
Epoch 13/1

**Insights**

During training,the neural network gradually minimizes prediction error.
As epochs increases,the model learns patterns from the dataset.
Lower validation loss indicates better learning performance.



**Model Evaluation**

In [18]:
loss,mae = model.evaluate(X_test_scaled,y_test)

print('Mean Absolute Error:',mae)
print('Loss:',loss)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 15.4049 - mae: 2.8620 
Mean Absolute Error: 2.862001895904541
Loss: 15.40488052368164


**Insights**

Lower loss and MAE values indicate better prediction performance.
The model evaluation helps measure how accurately the neural network predicts MPG values.

**Pedicting MPG Values**

In [19]:
y_pred = model.predict(X_test_scaled)

print(y_pred[:10])


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
[[31.042265]
 [19.78188 ]
 [33.682858]
 [29.099628]
 [25.748686]
 [29.985682]
 [12.84803 ]
 [29.529339]
 [20.521896]
 [36.589245]]


**Calculating the R2 Score**

In [20]:
r2 = r2_score(y_test,y_pred)

print('R2 Score:',r2)

R2 Score: 0.698183664539626


**Insights**

Here R2 Score = 0.70,
then the model explains 70% variation in MPG values.
Based on R2 Score,this is a good prediction model.

**Calculating MSE and RMSE**

---



In [21]:
mse = mean_squared_error(y_test,y_pred)
print('MSE:',mse)

rmse = np.sqrt(mse)
print('RMSE:',rmse)

MSE: 15.404881793216363
RMSE: 3.924905322834726


**Insights**

Here MSE is 15.40 ,that means predicted MPG values are closer to actual values.

RMSE is 3.92, that means better prediction accuracy.
RMSE is easier to interpret because it uses the same unit as the target variable.

**Creating Prediction DataFrame**

In [22]:
results = pd.DataFrame({
    'Actual_MPG':y_test,
    'Predicted_MPG':y_pred.flatten()
})
print(results.head(10))

     Actual_MPG  Predicted_MPG
79         26.0      31.042265
276        21.6      19.781879
248        36.1      33.682858
56         26.0      29.099628
393        27.0      25.748686
205        28.0      29.985682
43         13.0      12.848030
235        26.0      29.529339
152        19.0      20.521896
117        29.0      36.589245


**Saving prediction Output**

In [23]:
from google.colab import files

results.to_csv('mpg_predictions.csv',index=False)

files.download('mpg_predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>